# On‑Time AI: Modelos predictivos para reducir retrasos logísticos

**Autora:** Irene Díaz  
**Bootcamp:** Data Science  
**Fecha:** Mayo 2026  

---

## I. Introducción

### Contexto del problema

Una empresa internacional de comercio electrónico especializada en productos tecnológicos gestiona miles de envíos diarios desde un almacén central dividido en cinco bloques operativos. Su principal problema no es la capacidad de venta, sino la capacidad de entrega: casi seis de cada diez paquetes llegan tarde al cliente. En un sector donde la experiencia postventa determina la fidelización, un retraso no gestionado se convierte directamente en una reclamación, en una reseña negativa y, con el tiempo, en pérdida de reputación de marca.

El problema tiene una característica especialmente interesante desde el punto de vista analítico: **los retrasos son predecibles antes de que ocurran**. Los datos del historial de envíos contienen señales claras sobre qué paquetes van a llegar tarde. Si el modelo es capaz de identificar esos envíos en el momento del empaquetado o la expedición, la empresa puede activar protocolos preventivos: informar al cliente con antelación, enviar un cupón de compensación antes de que reclame, o reasignar recursos logísticos donde más se necesitan.

Este proyecto nace de esa premisa: no se trata de analizar los retrasos después de que hayan ocurrido, sino de anticiparlos para reducir su impacto.

### Objetivos y alcance

El objetivo principal de este proyecto es construir un modelo de Machine Learning capaz de predecir si un envío llegará tarde antes de que salga del almacén, con el máximo nivel de sensibilidad posible hacia los casos de retraso real. Esto implica que la métrica más importante no es el porcentaje de aciertos totales (Accuracy), sino el Recall: de todos los paquetes que realmente van a llegar tarde, ¿cuántos consigue detectar el modelo? Un retraso no detectado es un cliente que reclama sin haber recibido ninguna atención proactiva. Ese es el error que más le cuesta a la empresa.

Como objetivo secundario, el proyecto busca identificar qué variables del proceso logístico tienen mayor influencia en los retrasos, para que la empresa pueda tomar decisiones operativas concretas más allá del modelo. Por último, se aplica un análisis de clustering no supervisado para descubrir si existen perfiles naturales de envío con distintos niveles de riesgo, completando así una visión tanto predictiva como descriptiva del problema.

---

## II. Dataset

### Descripción de los datos

El dataset utilizado es **Shipping Data**, disponible públicamente en Kaggle. Contiene **10.999 registros** y **12 variables** que describen características operativas, financieras y de cliente para cada envío gestionado por la empresa.

| Variable | Tipo | Descripción |
|---|---|---|
| `ID` | Numérico | Identificador del cliente (descartado en el modelado) |
| `Warehouse_block` | Categórico | Bloque del almacén de origen (A, B, C, D, F) |
| `Mode_of_Shipment` | Categórico | Modo de transporte (Ship, Flight, Road) |
| `Customer_care_calls` | Numérico | Llamadas realizadas al servicio de atención |
| `Customer_rating` | Numérico | Valoración del cliente (1 peor – 5 mejor) |
| `Cost_of_the_Product` | Numérico | Precio del producto en USD |
| `Prior_purchases` | Numérico | Número de compras anteriores del cliente |
| `Product_importance` | Categórico ordinal | Importancia del producto (low, medium, high) |
| `Gender` | Categórico | Sexo del cliente (M/F) |
| `Discount_offered` | Numérico | Descuento aplicado al producto (%) |
| `Weight_in_gms` | Numérico | Peso del paquete en gramos |
| `Reached.on.Time_Y.N` | **Target** | **1 = retraso, 0 = entrega puntual** |

Es importante señalar que la variable target está codificada de forma contraintuitiva: el valor **1 indica retraso** y el **0 indica entrega a tiempo**. Esta convención se mantiene a lo largo de todo el proyecto.

La calidad del dataset es alta: no se detectaron valores nulos ni registros duplicados, lo que permitió pasar directamente al análisis exploratorio sin necesidad de imputaciones.

---


### Análisis Exploratorio (EDA)

El análisis exploratorio reveló varios hallazgos de gran valor para el negocio, que después resultaron confirmados matemáticamente por los modelos.

**La distribución del target** es el primer dato crítico: el **59.7% de los envíos llegan tarde**. Esto tiene dos implicaciones directas. La primera es de negocio: el problema es sistémico, no puntual. La segunda es metodológica: un modelo que dijera siempre «va a llegar tarde» acertaría el 59.7% de las veces sin haber aprendido nada. Por eso el Accuracy por sí solo no sirve como métrica, y hay que poner el foco en el Recall.

![Distribución del target](resources/img/20_distribución_de_entregas.png)

**El hallazgo más potente del EDA** es el comportamiento del descuento. El gráfico de barras por tramos muestra el patrón con una claridad contundente: en los tramos de 0-5% y 6-10% conviven envíos puntuales y tardíos, lo que indica que el descuento bajo no determina por sí solo el resultado. Sin embargo, a partir del tramo 15%, la barra verde desaparece completamente. Ningún envío con un descuento superior al 15% llega a tiempo. El umbral crítico está entre el 10% y el 15%, y a partir de ahí el retraso es prácticamente inevitable. La hipótesis de negocio es clara: las campañas promocionales agresivas saturan la capacidad logística del almacén hasta el punto de colapsar la puntualidad de forma sistemática.

![Distribución de descuentos por estado de entrega](resources/img/21_descuentos_retrasos.png)


**El peso del paquete** (`Weight_in_gms`) presenta un patrón diferente pero igual de relevante. El pairplot global muestra la existencia de dos «islas» horizontales muy marcadas en los datos: una concentración de paquetes ligeros (1-2 kg) con alta tasa de retraso, y otra de paquetes pesados (4-6 kg) con mayor mezcla de resultados. Esta segmentación natural por peso fue uno de los argumentos que justificaron el análisis de clustering del Notebook 03, donde el K-Means captura exactamente estos grupos sin haber visto nunca la etiqueta de retraso.

**El modo de envío** muestra que el barco (Ship) concentra el mayor volumen de incidencias en términos absolutos. No solo es el medio más utilizado (7.462 registros frente a los 3.537 restantes entre avión y carretera), sino que su proporción de retrasos es la más alta. Esto apunta a un problema estructural en la gestión de la carga marítima, posiblemente relacionado con tiempos de carga, procesos de aduana o saturación portuaria.

**La matriz de correlación** confirma que `Discount_offered` es la variable con mayor correlación con el target (0.40), seguida de `Weight_in_gms` con una correlación negativa (-0.27). La ausencia de correlaciones muy altas en el resto de variables sugiere que el problema es multifactorial y no lineal, lo que justifica el uso de algoritmos complejos de Machine Learning frente a modelos lineales simples.

**Respecto a los outliers**, el análisis de boxplots reveló que los descuentos altos son outliers para la clase puntual pero completamente normales para la clase retraso. Eliminarlos habría significado borrar exactamente la señal más importante del dataset, por lo que se tomó la decisión deliberada de mantenerlos.

## III. Preprocesamiento de los datos

### Limpieza y transformaciones

El preprocesamiento partió de un dataset ya limpio en términos de nulos y duplicados, por lo que el trabajo se centró en transformar las variables para que los algoritmos de Machine Learning pudieran interpretarlas correctamente.

La columna `ID` fue eliminada desde el primer paso. Es un identificador sin ningún valor predictivo y su presencia en el modelo no solo no aportaría nada, sino que podría introducir ruido artificial.

Para la variable `Product_importance` se aplicó **Ordinal Encoding** manual, mapeando 'low → 1', 'medium → 2', 'high → 3'. La razón de no usar One-Hot Encoding aquí es que esta variable tiene un orden lógico inherente: la importancia alta no es simplemente «diferente» de la baja, es cuantitativamente mayor. Preservar esa jerarquía tiene sentido tanto matemáticamente como desde el punto de vista del negocio.

Para `Warehouse_block`, `Mode_of_Shipment` y `Gender` se aplicó **One-Hot Encoding** con 'drop_first=True'. Estas tres variables son nominales, sin orden lógico entre sus categorías, por lo que la codificación ordinal no sería apropiada. El parámetro 'drop_first=True' elimina una columna por variable para evitar la multicolinealidad perfecta, que generaría problemas especialmente en la Regresión Logística. Las columnas resultantes de tipo booleano fueron convertidas a enteros para garantizar compatibilidad con todos los algoritmos de sklearn.


| Paso | Variable(s) | Decisión | Motivo |
|---|---|---|---|
| Eliminación | `ID` | Drop | Identificador sin valor predictivo |
| Ordinal Encoding | `Product_importance` | low=1, medium=2, high=3 | Preserva la jerarquía lógica |
| One-Hot Encoding | `Warehouse_block`, `Mode_of_Shipment`, `Gender` | get_dummies con drop_first=True | Sin orden natural, evita multicolinealidad |
| Conversión de tipos | Columnas booleanas | astype(int) | Compatibilidad con sklearn |
| Escalado | Todas las variables | StandardScaler | Necesario para Regresión Logística y KNN |


### División train/test

El dataset procesado fue dividido en un conjunto de entrenamiento (80%) y uno de test (20%), resultando en 8.799 muestras para entrenamiento y 2.200 para evaluación. Esta proporción responde a un equilibrio clásico en proyectos de este volumen: el 80% proporciona suficiente datos para que los modelos aprendan patrones generales, mientras que el 20% reserva una muestra representativa para evaluar si ese aprendizaje se generaliza a datos nuevos.

El parámetro 'stratify=y' fue crucial en esta división. Dado que el target está desequilibrado (59.7% retrasos frente a 40.3% envíos puntuales), una división aleatoria simple podría generar conjuntos con proporciones distintas, lo que haría la evaluación poco fiable. Con stratify, ambos conjuntos mantienen exactamente la misma distribución 60/40, garantizando que el modelo aprende y es evaluado sobre la misma realidad estadística. La semilla 'random_state=42' asegura que la división es reproducible por cualquier persona que clone el repositorio.

Los splits fueron exportados a `data/train.csv` y `data/test.csv` para que el experimento sea completamente reproducible sin necesidad de volver a ejecutar el preprocesamiento.

---

## IV. Modelado supervisado

### Estrategia de evaluación

Antes de entrenar ningún modelo, fue necesario definir qué significa «un buen modelo» para este problema concreto. La respuesta es importante, porque distintas métricas cuentan historias muy distintas y pueden llevar a conclusiones opuestas.

En este proyecto, **la métrica prioritaria es el Recall** (también llamado sensibilidad o tasa de verdaderos positivos). El Recall responde a la pregunta: de todos los paquetes que realmente van a llegar tarde, ¿qué porcentaje detecta el modelo? Un Recall del 60% significa que 4 de cada 10 retrasos reales pasan desapercibidos: esos clientes recibirán su paquete tarde sin haber recibido ninguna comunicación preventiva, y reclamarán enfadados. Un Recall del 90% significa que solo 1 de cada 10 se escapa.

El Accuracy, en cambio, puede ser engañoso aquí. Un modelo que dijera siempre «retraso» obtendría un Accuracy del 59.7% sin haber aprendido nada. Por eso se reportan también la Precision (de las alertas lanzadas, ¿cuántas eran reales?), el F1-Score (media armónica entre Precision y Recall) y el ROC-AUC (capacidad discriminativa general del modelo en todos los umbrales posibles).

Se entrenaron **cinco modelos**, cada uno representando una familia distinta de algoritmos de clasificación, para tener una comparativa completa y fundamentar la elección final con evidencia sólida.

| Modelo | Familia |
|---|---|
| Regresión Logística | Modelos lineales |
| Árbol de Decisión | Reglas interpretables |
| Random Forest | Ensamble en paralelo (bagging) |
| XGBoost | Ensamble en serie (boosting) |
| KNN | Similitud entre casos |

---


### Modelo 1: Regresión Logística

La Regresión Logística actúa como **modelo de referencia**. Es el algoritmo más simple de clasificación binaria: aprende el peso de cada variable y los combina linealmente para calcular una probabilidad de retraso. Si esa probabilidad supera el 50%, predice retraso.

Dado que necesita que todas las variables estén en la misma escala numérica (un peso de 5.000g dominaría sobre un descuento de 10% sin normalización), se utilizó un Pipeline que encadena primero un StandardScaler y luego el modelo. El escalado se aprende exclusivamente sobre el conjunto de train y se aplica igual al test, evitando cualquier contaminación de información.

| Métrica | Resultado |
|---|---|
| Accuracy | 64.0% |
| Precision | 70.9% |
| Recall | 67.4% |
| F1-Score | 69.1% |
| ROC-AUC | 0.717 |

El modelo mejora el baseline trivial (59.7%) pero deja escapar 428 retrasos reales de los 1.313 existentes en el test. Es un punto de partida sólido y cualquier modelo más complejo deberá superarlo para justificar su uso.

![Matriz de confusión de Regresión Logística](resources/img/01_matriz_confusion_reg_log.png)

Estos 428 falsos negativos son el punto de partida de la historia. A partir de aquí, cada modelo que se pruebe tiene un objetivo claro: reducir ese número sin disparar las falsas alarmas. El recorrido desde estos 428 hasta los 26 del modelo final es, en buena medida, el argumento central de este proyecto.

---

### Modelo 2: Árbol de Decisión

El Árbol de Decisión es el modelo más interpretable: su «razonamiento» puede visualizarse como un diagrama de flujo de preguntas encadenadas. Aprende automáticamente cuáles son las mejores preguntas y en qué orden hacerlas para separar retrasos de envíos puntuales.

Se limitó la profundidad a `max_depth=5` para evitar el overfitting: sin ese límite, el árbol memorizaría los datos de entrenamiento en lugar de aprender patrones generales. El check de overfitting confirmó que la diferencia entre Accuracy en train (69.2%) y en test (68.0%) es de solo 1.2 puntos, lo que indica que el modelo generaliza correctamente.

| Métrica | Resultado |
|---|---|
| Accuracy | 68.0% |
| Precision | 96.9% |
| Recall | 47.8% |
| F1-Score | 64.0% |
| ROC-AUC | 0.736 |

La visualización del árbol revela algo muy valioso: la **primera pregunta que hace el modelo es `Discount_offered <= 10.5`**, confirmando matemáticamente el hallazgo del EDA. La rama derecha (descuento alto) tiene gini = 0.0: los 2.126 paquetes con descuento alto son TODOS retrasos, sin una sola excepción, de ahí la Precision del 96.9%.

Sin embargo, el Recall del 47.8% es inaceptable para nuestro objetivo: el modelo solo detecta menos de la mitad de los retrasos reales. Deja escapar 685 clientes. El árbol es demasiado conservador: solo alerta cuando está prácticamente seguro al 100%, ignorando todos los retrasos que no pasan por la rama del descuento alto.

---

### Modelo 3: Random Forest

El Random Forest construye 100 árboles de decisión de forma independiente, cada uno entrenado sobre una muestra aleatoria distinta del dataset y con acceso a un subconjunto aleatorio de variables en cada nodo. La predicción final es la votación mayoritaria de los 100 árboles. Este mecanismo de **bagging** corrige el problema de conservadurismo del árbol simple: donde un solo árbol se callaba, 100 árboles votando en conjunto se atreven a lanzar la alerta.

| Métrica | Resultado |
|---|---|
| Accuracy | 65.9% |
| Precision | 76.4% |
| Recall | 61.9% |
| F1-Score | 68.4% |
| ROC-AUC | 0.735 |

El Recall sube al 61.9%, detectando 813 retrasos frente a los 628 del árbol simple: 185 clientes más que reciben el cupón preventivo. La Precision baja al 76.4%, lo que significa que se generan más falsas alarmas (251 cupones enviados de más), pero ese es un intercambio razonable: el coste de un cupón innecesario es siempre menor que el daño de reputación de un cliente que reclama sin haber sido atendido.

El análisis de **importancia de variables** del Random Forest es uno de los resultados más valiosos del proyecto. Las tres variables que concentran el 68% del poder predictivo son:
- `Weight_in_gms` (≈28%): el peso del paquete es el factor más determinante.
- `Discount_offered` (≈23%): confirmado por el EDA y el árbol de decisión.
- `Cost_of_the_Product` (≈17%): productos de mayor valor requieren procesos especiales que ralentizan el envío.

![Importancia de las variables en Random Forest](resources/img/05_importancia_variables_random_forest.png)

---

### Modelo 4: XGBoost

XGBoost construye árboles en **serie** en lugar de en paralelo: cada árbol nuevo no parte de cero, sino que se especializa en corregir los errores del anterior. Este mecanismo de **boosting** acumula el aprendizaje iterativamente, haciendo al modelo muy eficaz en datasets con patrones complejos.

Se eligió XGBoost sobre el Gradient Boosting clásico de sklearn por tres razones: su regularización nativa (L1 y L2) previene el overfitting, el parámetro `scale_pos_weight` permite compensar el desequilibrio de clases de forma nativa, y su velocidad lo hace más eficiente para búsquedas de hiperparámetros.

**El primer ajuste clave del proyecto** ocurrió aquí. El parámetro `scale_pos_weight` le indica a XGBoost qué clase es más importante en términos de error. La fórmula correcta para maximizar el Recall es `positivos / negativos` (retrasos / envíos a tiempo) ≈ 1.48: con este valor, el modelo penaliza más los errores sobre los retrasos, que son precisamente los que nos importa detectar. Un error en la dirección contraria (negativos/positivos ≈ 0.68) hace exactamente lo opuesto: prioriza los envíos puntuales y produce un Recall desastroso del 47%. Un solo parámetro. 44 puntos de diferencia en Recall.

**El segundo ajuste clave** fue el `learning_rate`, optimizado mediante GridSearchCV. Se exploraron 8 combinaciones de hiperparámetros con validación cruzada de 5 folds, optimizando por Recall. El resultado: `learning_rate=0.05` frente al 0.1 inicial. Una tasa de aprendizaje más baja significa que cada árbol corrige el error de forma más conservadora y precisa. El Recall sube del 91.6% al 98.0%. Otro ajuste de parámetro, otros 6 puntos más de Recall.

| Métrica | Resultado |
|---|---|
| Accuracy | 59.8% |
| Precision | 60.0% |
| **Recall** | **98.0%** |
| F1-Score | 74.4% |
| ROC-AUC | 0.756 |

Con ambos ajustes aplicados, XGBoost detecta **98 de cada 100 retrasos reales**. Solo 26 clientes de los 1.313 se escapan sin atención preventiva, frente a los 685 que dejaba escapar el árbol de decisión o los 500 del Random Forest. La Precision más baja (60%) implica más cupones enviados de más (859 falsos positivos), pero ese intercambio está perfectamente alineado con el objetivo de negocio.

![Matriz de confusión de XGBoost](resources/img/07_matriz_confusion_xgb.png)

Se realizó también un análisis de ajuste de umbral para verificar si bajando el umbral de decisión del 50% por defecto se podía mejorar aún más el equilibrio. La tabla de resultados mostró que a partir de umbral 0.40, el Recall sube al 99-100% pero la Precision cae drásticamente y se mantiene plana: el modelo prácticamente etiqueta todo como retraso, lo que no aporta valor discriminativo real. El **umbral óptimo es 0.50**, el valor por defecto, que con los ajustes aplicados ya ofrece el mejor equilibrio posible.


---

### Modelo 5: KNN (K-Nearest Neighbors)

KNN es el modelo más intuitivo: para predecir si un paquete llegará tarde, busca los 5 paquetes más similares del conjunto de entrenamiento y vota según lo que les ocurrió a ellos. La «similitud» se mide como distancia matemática, por lo que necesita escalado (mismo Pipeline que la Regresión Logística).

| Métrica | Resultado |
|---|---|
| Accuracy | 62.3% |
| Precision | 69.6% |
| Recall | 65.4% |
| F1-Score | 67.5% |
| ROC-AUC | 0.691 |

KNN es el modelo más débil del grupo en términos globales. Su ROC-AUC de 0.691 es el más bajo con diferencia, y su Accuracy apenas supera el baseline trivial. Tiene un Recall aceptable (65.4%) pero lo consigue generando 375 falsas alarmas, el número más alto de todos. Su valor en este proyecto es fundamentalmente didáctico: demuestra que la simplicidad conceptual no se traduce en buen rendimiento cuando hay muchas variables y los datos son dispersos.

Los modelos basados en árboles como Random Forest y XGBoost funcionan mejor en este tipo de datos porque no miden distancias: aprenden reglas del tipo «si el descuento supera el 10% y el peso es inferior a 4.000g, predice retraso». Esas reglas son robustas frente a variables en distintas escalas, a relaciones no lineales y a la presencia de variables poco relevantes. KNN, en cambio, trata todas las variables por igual al calcular la distancia, lo que hace que variables con poco poder predictivo como el bloque del almacén o el género del cliente contaminen la búsqueda de vecinos y diluyan la señal de las que realmente importan.

---


### Elección del modelo

La tabla resumen de los cinco modelos muestra con claridad la evolución del aprendizaje a lo largo del notebook:

| Modelo | Accuracy | Precision | Recall | F1-Score | ROC-AUC | FN |
|---|---|---|---|---|---|---|
| Regresión Logística | 64.0% | 70.9% | 67.4% | 69.1% | 0.717 | 428 |
| Árbol de Decisión | 68.0% | 96.9% | 47.8% | 64.0% | 0.736 | 685 |
| Random Forest | 65.9% | 76.4% | 61.9% | 68.4% | 0.735 | 500 |
| **XGBoost** | **59.8%** | **60.0%** | **98.0%** | **74.4%** | **0.756** | **26** |
| KNN | 62.3% | 69.6% | 65.4% | 67.5% | 0.691 | 454 |

![Comparativa 5 modelos](resources/img/08_comparativa_5_modelos.png)

**El modelo elegido es XGBoost** optimizado mediante GridSearch con `scale_pos_weight` = positivos/negativos ≈ 1.48 y `learning_rate` = 0.05. Gana en Recall (98.0%), F1-Score (74.4%) y ROC-AUC (0.756). En la métrica que más importa para este negocio supera al segundo clasificado (Random Forest, 61.9%) por más de 36 puntos porcentuales. Los 26 falsos negativos son el resultado más tangible del proyecto: de los 1.313 retrasos reales del conjunto de test, solo 26 se escapan sin atención preventiva.

La lección más importante de este proyecto de modelado no es qué algoritmo gana, sino que **alinear los parámetros del modelo con el objetivo de negocio es tan determinante como la elección del algoritmo**. Un XGBoost mal configurado es el peor modelo del grupo. Un XGBoost correctamente configurado y optimizado es el mejor por un margen amplio.

---

## V. Clustering no supervisado

### K-Means

Complementando el modelo supervisado, se aplicó un análisis de clustering K-Means sobre las 7 variables numéricas del dataset (excluyendo el target y las variables categóricas sin codificar, para que el modelo agrupara por perfil de riesgo y no por bloque de almacén o modo de transporte).

El número óptimo de clusters fue determinado mediante tres métodos independientes: el Método del Codo, el Índice de Silhouette y el diagrama de Silhouette comparativo. Los tres convergieron en **K = 3**, con un Silhouette Score de 0.2386 (el máximo del rango explorado). Bien es cierto que el Método del Codo no muestra un punto de inflexión dramático, pero el Índice de Silhouette tiene su pico máximo en K=3 **(0.2386)**, cayendo notablemente en K=4 (0.193). Por otro lado, el diagrama de Silhouette confirma que K=3 es el único K donde los tres bloques son simultáneamente equilibrados y mayoritariamente por encima de la media global.

![Gráficos codo y silhouette](resources/img/10_codo_silhouette.png)


El hallazgo más relevante del clustering es que el modelo encontró espontáneamente, sin ver ninguna etiqueta de retraso, los mismos patrones que los modelos supervisados habían identificado:

| Cluster | Tamaño | Tasa de retraso | Perfil dominante |
|---|---|---|---|
| 🟣 Cluster 0 | 2.294 (20.9%) | **99.5%** | Descuento alto (media 40.1%) |
| 🟢 Cluster 1 | 6.097 (55.4%) | 47.9% | Paquete pesado (media 4.801g) |
| 🔵 Cluster 2 | 2.608 (23.7%) | 52.3% | Cliente fidelizado (compras previas: 5.1) |

![Resultados de K-Means](resources/img/15_resultados_clustering_kmeans_k3.png)

El Cluster 0 es el resultado más impactante del proyecto: el K-Means agrupa el 20.9% de los envíos en un perfil cuya tasa de retraso es del 99.5%. Este grupo se define exclusivamente por el descuento alto, con un peso y un coste del producto por debajo de la media. La conclusión es la misma que señalaron el EDA y el árbol de decisión, pero ahora desde una metodología completamente independiente: las campañas promocionales agresivas son el factor de riesgo número uno de la cadena logística.

Este análisis complementa al modelo supervisado: mientras XGBoost predice si un envío concreto llegará tarde, el K-Means identifica qué perfiles estructurales tienen mayor riesgo sistémico. Usados conjuntamente forman un sistema de alerta más completo: el clustering actúa a nivel estratégico (segmentos de riesgo) y el modelo supervisado actúa a nivel operativo (envío individual).

---

## VI. Predicción y resultados finales

### El modelo en producción

El modelo final es un **XGBClassifier** entrenado con 100 árboles en serie, una tasa de aprendizaje de 0.05 y árboles de profundidad máxima 3 para evitar el overfitting característico del boosting. El parámetro más crítico es 'scale_pos_weight ≈ 1.48', que le indica al modelo que detectar un retraso vale más que evitar una falsa alarma, alineando su comportamiento interno con el objetivo de negocio desde el entrenamiento. El modelo ha sido serializado con pickle y guardado en 'model/production/xgboost_final.pkl', listo para ser cargado y usado en producción sin necesidad de reentrenar.

### Resultados sobre el conjunto de test

| | Predicho: A tiempo | Predicho: Retraso |
|---|---|---|
| **Real: A tiempo** | 28 (TN) | 859 (FP) |
| **Real: Retraso** | 26 (FN) | 1.287 (TP) |

La evaluación se realizó sobre 2.200 registros que el modelo nunca había visto durante el entrenamiento. De los 1.313 retrasos reales presentes en ese conjunto, el modelo detecta correctamente 1.287. Solo 26 se escapan sin atención preventiva. Al mismo tiempo, lanza 859 alertas sobre envíos que finalmente llegaron a tiempo: son cupones enviados de más, un coste asumible y predecible.

El Recall del 98.0% es el número que mejor resume el rendimiento del modelo: 98 de cada 100 clientes que van a sufrir un retraso reciben el cupón preventivo antes de que su paquete llegue tarde. Solo 2 de cada 100 reclaman sin haber sido atendidos. El modelo también arroja un Accuracy del 59.8% y un ROC-AUC de 0.756, confirmando que su capacidad discriminativa general es la más alta de todos los modelos probados.

### Impacto de negocio

Un retraso no gestionado genera una cadena de consecuencias que va mucho más allá del incidente puntual: reclamación al servicio de atención, reseña negativa, y en muchos casos pérdida del cliente. Un cupón enviado de más, en cambio, sorprende positivamente a un cliente cuyo paquete llega a tiempo. El intercambio no es solo aceptable, es exactamente el que una empresa orientada a la experiencia del cliente debería buscar.

Con este modelo, la empresa pasa de gestionar los retrasos de forma reactiva a hacerlo de forma proactiva, anticipándose antes de que el cliente tenga motivo para quejarse, con un impacto directo sobre la reputación de marca y la tasa de retención. Traducido a números: de cada 100 clientes que van a sufrir un retraso, 98 reciben el cupón preventivo antes de que su paquete llegue tarde. El coste son los 859 cupones enviados de más sobre 10.999 envíos, un volumen asumible y predecible. El beneficio es eliminar el 98.0% de las reclamaciones sorpresivas, que son las que realmente dañan la reputación de una marca.

---

## VII. Conclusiones

Este proyecto ha demostrado que los retrasos logísticos de esta empresa no son aleatorios: tienen causas identificables, medibles y, lo más importante, anticipables con Machine Learning.

El hallazgo central, que aparece confirmado desde tres metodologías independientes (EDA, árbol de decisión supervisado y clustering no supervisado), es que los descuentos superiores al 10% generan un colapso operativo casi garantizado. Las campañas promocionales saturan la capacidad logística del almacén de una forma que el resto de factores no puede compensar. Este patrón no es una anomalía estadística: es una señal robusta y reproducible que debe traducirse en decisiones operativas concretas.

El modelo XGBoost, con dos ajustes consecutivos bien razonados, alcanza un Recall del 98.0%: 98 de cada 100 retrasos pueden ser anticipados antes de que el cliente los sufra. El primer ajuste fue corregir el `scale_pos_weight` de 0.68 a 1.48, alineando el modelo con el objetivo de negocio. El segundo fue optimizar el `learning_rate` a 0.05 mediante GridSearchCV. Ninguno requirió cambiar el algoritmo ni añadir más datos. Solo entender qué le estábamos pidiendo al modelo y configurarlo en consecuencia. Es probablemente la lección más transferible de todo el proyecto: los modelos no son neutrales, y su configuración debe reflejar conscientemente el coste relativo de cada tipo de error.

### Siguientes pasos

En cuanto a los pasos futuros, el paso más inmediato y de mayor impacto con menor esfuerzo sería añadir la variable `Cluster` generada por el K-Means como feature adicional al modelo supervisado. El Cluster 0 concentra el 20.9% de los envíos con una tasa de retraso del 99.5%. Decirle al XGBoost a qué cluster pertenece cada envío antes de que prediga es darle una señal casi perfecta para ese grupo. Es una mejora que se puede implementar directamente sobre el trabajo ya realizado en el Notebook 03.

En un plazo más largo, enriquecer el dataset con variables externas sería el salto cualitativo más grande. Hoy el modelo predice con lo que sabe en el momento del empaquetado: el peso, el descuento, el modo de envío. Si incorporara datos de capacidad del almacén en tiempo real, calendario de campañas promocionales, condiciones meteorológicas o historial de incidencias por ruta, el modelo pasaría de anticipar retrasos a explicar exactamente por qué van a ocurrir, lo que abre la puerta a intervenciones operativas mucho más precisas.

Como complemento práctico al modelo, se ha desarrollado una aplicación interactiva con Streamlit que permite realizar predicciones individuales, procesar envíos por lotes mediante carga de CSV, y explorar visualmente los principales hallazgos del proyecto. La app está disponible en 'src/app.py'

---